In [0]:
%run ../control_framework/__init__

In [0]:
%sql
select * from ni_m101_finnhub_dev.finnhubb_daily_stock_price_ts

In [0]:
import requests

In [0]:
def get_finnhub_connection(symbol,type):
    url = f"https://finnhub.io/api/v1/{type}"
    params = {
        "symbol": symbol,
        "token": 'd9d7mc9r01qj7sqbbs2gd9d7mc9r01qj7sqbbs30'
    }
    response = requests.get(url, params=params)
    return response.json()

In [0]:
import json
df = spark.createDataFrame(get_finnhub_connection('AAPL','/quote'))
display(df)

In [0]:
def data_list(symbol_list):
    symbol_count = len(symbol_list)
    data_list = []
    for i in range(symbol_count):
        if i % 50 == 0 and i != 0:
            time.sleep(60)  # add one minute buffer
        data = get_finnhub_connection(symbol_list[i],'quote')
        data['symbol'] = symbol_list[i]
        data['process_date'] = 20260728
        data['r_source'] = 'm101_finnhub'
        # print(data)
        hash_id = create_hash_id(data)
        data['hash_id'] = hash_id
        # print(data)
        data_list.append(data)
    return data_list
# data_list = data_list(['MMM','ABT'])
# print(data_list)

In [0]:
data = data_list(['MMM','ABT'])
df = spark.createDataFrame(data)
df = df.withColumn("process_date", df["process_date"].cast("string")).withColumn("t", df["t"].cast("int"))
display(df)


In [0]:
%sql
select *,cast(process_date as int)  from temp_finnhub_quotes

In [0]:
# Validation rules: check column types, store anomalies as all-string rows
from pyspark.sql.types import StringType, DoubleType, IntegerType

expected_schema = {
        "symbol": StringType(),
        "process_date": StringType(),
        "r_source": StringType(),
        "hash_id": StringType(),
        "c": DoubleType(),
        "d": DoubleType(),
        "h": DoubleType(),
        "l": DoubleType(),
        "o": DoubleType(),
        "pc": DoubleType(),
        "t": IntegerType(),
        'dp':DoubleType(),
    }

def validate_and_store_anomalies(df, expected_schema):
    from pyspark.sql.types import StructType, StructField
    from pyspark.sql.functions import col

    # Create all-string schema for anomaly table
    string_schema = StructType([StructField(col_name, StringType(), True) for col_name in expected_schema.keys()])

    # Build filter expression for type mismatches
    mismatch_expr = []
    for col_name, dtype in expected_schema.items():
        if isinstance(dtype, StringType):
            mismatch_expr.append(~col(col_name).cast(StringType()).isNull() & ~col(col_name).cast(StringType()).eqNullSafe(col(col_name)))
        elif isinstance(dtype, DoubleType):
            mismatch_expr.append(~col(col_name).cast(DoubleType()).isNull() & ~col(col_name).cast(DoubleType()).eqNullSafe(col(col_name)))
        elif isinstance(dtype, IntegerType):
            mismatch_expr.append(~col(col_name).cast(IntegerType()).isNull() & ~col(col_name).cast(IntegerType()).eqNullSafe(col(col_name)))
    from functools import reduce
    filter_condition = reduce(lambda a, b: a | b, mismatch_expr)

    anomaly_df = df.filter(filter_condition)
    # Cast all columns to string for anomaly table
    for col_name in expected_schema.keys():
        anomaly_df = anomaly_df.withColumn(col_name, col(col_name).cast(StringType()))
    anomaly_df = anomaly_df.select(*expected_schema.keys())
    anomaly_df.createOrReplaceTempView("anomaly_finnhub_quotes")
    display(anomaly_df)
    return anomaly_df

def remove_anomalies_by_hash_id(df, anomaly_df):
    from pyspark.sql.functions import col
    anomaly_hash_ids = anomaly_df.select("hash_id").distinct()
    df_clean = df.join(anomaly_hash_ids, on="hash_id", how="left_anti")
    return df_clean

def insert_into_target_table(df, target_table):
    # Insert new rows into target table, preserving previous data
    df.write.format("delta").mode("append").saveAsTable(target_table)

def create_target_table(table_name, expected_schema):
    from pyspark.sql.types import StructType, StructField
    fields = [StructField(col_name, dtype, True) for col_name, dtype in expected_schema.items()]
    schema = StructType(fields)
    empty_df = spark.createDataFrame([], schema)
    empty_df.write.format("delta").mode("overwrite").saveAsTable(table_name)

validate_and_store_anomalies(df, expected_schema)

In [0]:
create_target_table('ni_m101_finnhub_dev.test_1',expected_schema)


In [0]:
anomoly_df = validate_and_store_anomalies(df, expected_schema)


In [0]:
cleaned_df = remove_anomalies_by_hash_id(df,anomoly_df)

In [0]:
insert_into_target_table(cleaned_df,'ni_m101_finnhub_dev.test_1')

In [0]:
%sql
select * from ni_m101_finnhub_dev.test_1

In [0]:
validate_and_store_anomalies(df, expected_schema)